In [1]:
!pip install presidio-analyzer presidio-anonymizer Faker PyMuPDF spacy tqdm
!python -m spacy download en_core_web_trf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.0/259.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
  Attempting uninstall: spacy
    Found existing installation: spacy 3.8.14
    Uninstalling spacy-3.8.14:
      Successfully uninstalled spacy-3.8.14
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

In [2]:
import fitz  # PyMuPDF
import tqdm
from faker import Faker
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider

print("Step 1: Extracting text from PDF...")
pdf_path = "Red Herring Prospectus.pdf"
doc = fitz.open(pdf_path)

print("Step 2: Initializing Generalized NLP Pipeline (Transformer)...")

configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_trf"}]
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine_transformer = provider.create_engine()

registry = RecognizerRegistry()
registry.load_predefined_recognizers()


universal_phone_regex = r"(?:\+?\s*91\s*)?(?:0\s*)?[1-9](?:[\s\n\-\.]*\d){7,13}"
robust_phone_recognizer = PatternRecognizer(
    supported_entity="PHONE_NUMBER",
    patterns=[Pattern(name="universal_indian_phone", regex=universal_phone_regex, score=0.85)]
)
registry.add_recognizer(robust_phone_recognizer)

#  the Analyzer
analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine_transformer,
    registry=registry
)

print("Step 3: Initializing Anonymizer and Consistency Memory...")
fake = Faker('en_IN')
pii_mapping = {}

def get_fake_value(entity_type, original_text):
    # Standardize the key by stripping leading/trailing whitespace and newlines for memory lookup
    clean_original = original_text.strip()

    if clean_original in pii_mapping:
        return pii_mapping[clean_original]

    if entity_type == 'PERSON':
        fake_val = fake.name()
    elif entity_type == 'EMAIL_ADDRESS':
        fake_val = fake.email()
    elif entity_type == 'PHONE_NUMBER':
        fake_val = fake.phone_number()
    elif entity_type == 'LOCATION':
        fake_val = fake.city()
    elif entity_type == 'ORG':
        fake_val = fake.company()
    else:
        fake_val = f"[{entity_type}_REDACTED]"

    pii_mapping[clean_original] = fake_val
    return fake_val

print("Step 4: Starting Full Document Processing...")
# The Allow-List to prevent over-redaction of legal jargon
financial_jargon_allowlist = [
    "Equity Shares", "Red Herring Prospectus", "The Board", "Act",
    "Government of India", "State Government", "Registrar", "SEBI", "BSE", "NSE"
]
entities_to_find = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION", "ORG", "IP_ADDRESS"]

redacted_pages = []

# Process the document page by page
for page_num in tqdm.tqdm(range(len(doc))):
    page_text = doc.load_page(page_num).get_text("text")


    results = analyzer.analyze(text=page_text, entities=entities_to_find, language='en')


    filtered_results = []
    for r in results:
        extracted_text = page_text[r.start:r.end]
        # Keep the entity only if it is NOT in our jargon allow-list
        if not any(jargon.lower() in extracted_text.lower() for jargon in financial_jargon_allowlist):
            filtered_results.append(r)

    # 3. Sort backwards to avoid index shifting during replacement
    results_sorted = sorted(filtered_results, key=lambda x: x.start, reverse=True)

    # 4. Inject the fake data
    page_redacted = page_text
    for result in results_sorted:
        original_pii = page_text[result.start:result.end]
        fake_pii = get_fake_value(result.entity_type, original_pii)
        page_redacted = page_redacted[:result.start] + fake_pii + page_redacted[result.end:]

    redacted_pages.append(page_redacted)

# Join the pages and save
final_redacted_text = "\n".join(redacted_pages)
output_filename = "Redacted_Output.txt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(final_redacted_text)

print(f"\n Full document processed with generalized rules. Saved as {output_filename}")

Step 1: Extracting text from PDF...
Step 2: Initializing Generalized NLP Pipeline (Transformer)...


Step 3: Initializing Anonymizer and Consistency Memory...
Step 4: Starting Full Document Processing...


  0%|          | 0/128 [00:00<?, ?it/s]WARNING:presidio-analyzer:Entity ORG doesn't have the corresponding recognizer in language : en
/usr/local/lib/python3.12/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
100%|██████████| 128/128 [00:20<00:00,  6.28it/s]


✅ Success! Full document processed with generalized rules. Saved as Redacted_Output.txt


--- RUNNING QUANTITATIVE EVALUATION (GOLDEN TEST SET) ---

Ground Truth Count: 6
Predictions Count: 6


--- FINAL METRICS ---
Precision: 1.00 (How much of what we redacted was actually PII?)
Recall:    1.00 (How much of the total PII did we successfully catch?)
F1-Score:  1.00 (The harmonic mean of Precision and Recall)
